In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")
df.head(10)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1
5,referral,manufacturing,employed,africa,43841.0,3,5,0.77,1
6,NaN,education,employed,north_america,58738.0,3,9,0.78,1
7,paid_ads,technology,employed,north_america,95059.0,2,4,0.41,1
8,organic_search,healthcare,employed,asia,62565.0,3,6,0.52,1
9,referral,retail,student,asia,12000.0,2,2,0.35,0


In [3]:
categorical_cols = df.select_dtypes(include=['str', 'category']).columns.tolist()
print("categorical coumns:", categorical_cols)
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("numerical columns:", numerical_cols)

categorical coumns: ['lead_source', 'industry', 'employment_status', 'location']
numerical columns: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']


In [4]:
if 'converted' in categorical_cols:
    categorical_cols.remove('converted')
if 'converted' in numerical_cols:
    numerical_cols.remove('converted')
df[categorical_cols] = df[categorical_cols].fillna('NA')
df[numerical_cols] = df[numerical_cols].fillna(0.0)

In [5]:
# Q1 
industry_mode = df['industry'].mode()[0]
print(f"--- Question 1 ---")
print(f""" Mode (the most frequent observation) for the column "industry": {industry_mode}""")

--- Question 1 ---
 Mode (the most frequent observation) for the column "industry": technology


In [6]:
# Q2
corr_matrix = df[numerical_cols].corr()
print("Correlation matrix:\n", corr_matrix)

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]
print("\n--- Question 2 ---")
print("Correlation coefficients of the pairs:")
max_corr = -1
max_pair = None

for feat1, feat2 in pairs:
    if feat1 in corr_matrix.columns and feat2 in corr_matrix.columns:
        corr_val = abs(corr_matrix.loc[feat1, feat2])
        print(f"{feat1} & {feat2}: {corr_val:.4f}")
        if corr_val > max_corr:
            max_corr = corr_val
            max_pair = (feat1, feat2)

print(f"\nBiggest correlation: {max_pair[0]} and {max_pair[1]} (Correlation: {max_corr:.4f})\n")

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

Correlation matrix:
                           annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  

--- Question 2 ---
Correlation coefficients of the pairs:
interaction_count & lead_score: 0.9157
number_of_courses_viewed & lead_score: 0.7572
number_of_courses_viewed & interaction_count: 0.7216
annual_income & interaction_count: 0.1228

Biggest correlation: interaction_count and lead_score (Correlation: 0.9157)



In [7]:
# Q3
train_categorical_cols = df_train.select_dtypes(include=['str', 'category']).columns
def calculate_mi(series):
    return mutual_info_score(series, y_train)

mi_scores = df_train[train_categorical_cols].apply(calculate_mi)
mi_scores = mi_scores.round(2).sort_values(ascending=False)

print("--- Question 3 ---")
print("Mutual Information Scores of Categorical Variables:")
print(mi_scores)

question_variables = ['industry', 'location', 'lead_source', 'employment_status']
filtered_mi = mi_scores.loc[question_variables]
biggest_mi_present = filtered_mi.idxmax()

print(f"\nVariable having biggest MI score: {biggest_mi_present} ({filtered_mi[biggest_mi_present]})")

--- Question 3 ---
Mutual Information Scores of Categorical Variables:
lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

Variable having biggest MI score: lead_source (0.03)


In [8]:
# Q4
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

#Logistic Regression
model = LogisticRegression(
    solver='liblinear', 
    C=1.0, 
    max_iter=1000, 
    random_state=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

val_accuracy = accuracy_score(y_val, y_pred)
rounded_accuracy = round(val_accuracy, 2)

print("--- Question 4 ---")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Accuracy (rounded): {rounded_accuracy:.2f}")

--- Question 4 ---
Validation Accuracy: 0.6450
Validation Accuracy (rounded): 0.65


In [13]:
# Q5
original_accuracy = accuracy_score(y_val, y_pred)
print("--- Question 5 ---")
print(f"Orijinal Validation Accuracy: {original_accuracy:.6f}\n")

features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']
differences = {}
for feature in features_to_check:
    df_train_subset = df_train.drop(columns=[feature])
    df_val_subset = df_val.drop(columns=[feature])
    
    train_dicts_subset = df_train_subset.to_dict(orient='records')
    val_dicts_subset = df_val_subset.to_dict(orient='records')
    
    dv_subset = DictVectorizer(sparse=False)
    X_train_subset = dv_subset.fit_transform(train_dicts_subset)
    X_val_subset = dv_subset.transform(val_dicts_subset)
    
    model_subset = LogisticRegression(
        solver='liblinear', 
        C=1.0, 
        max_iter=1000, 
        random_state=42
    )
    model_subset.fit(X_train_subset, y_train)
    
    y_pred_subset = model_subset.predict(X_val_subset)
    subset_accuracy = accuracy_score(y_val, y_pred_subset)
    
    diff = original_accuracy - subset_accuracy
    differences[feature] = diff
    
    print(f"'{feature}' feature eliminated. Accuracy: {subset_accuracy:.6f} | Difference (original - new): {diff:.6f}")

smallest_feature = min(differences, key=differences.get)

print(f"\nSmallest feature: {smallest_feature} (Difference: {differences[smallest_feature]:.6f})")


--- Question 5 ---
Orijinal Validation Accuracy: 0.645000

'lead_source' feature eliminated. Accuracy: 0.642000 | Difference (original - new): 0.003000
'number_of_courses_viewed' feature eliminated. Accuracy: 0.643000 | Difference (original - new): 0.002000
'interaction_count' feature eliminated. Accuracy: 0.601000 | Difference (original - new): 0.044000

Smallest feature: number_of_courses_viewed (Difference: 0.002000)


In [14]:
# Q6
c_values = [0.000001, 0.00001, 0.0001, 0.001]
results = {}

print("--- Question 6 ---")
for c in c_values:
    model = LogisticRegression(
        solver='liblinear', 
        C=c, 
        max_iter=1000, 
        random_state=42
    )
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    rounded_acc = round(acc, 3)
    
    results[c] = rounded_acc
    print(f"C = {c:<8} -> Accuracy (Raw): {acc:.6f} | Accuracy (Rounded): {rounded_acc:.3f}")

max_acc = max(results.values())
best_c_candidates = [c for c, acc in results.items() if acc == max_acc]
best_c = min(best_c_candidates)

print(f"\nBest accuracy: ({max_acc:.3f}). Smallest c score: {best_c}")

--- Question 6 ---
C = 1e-06    -> Accuracy (Raw): 0.598000 | Accuracy (Rounded): 0.598
C = 1e-05    -> Accuracy (Raw): 0.598000 | Accuracy (Rounded): 0.598
C = 0.0001   -> Accuracy (Raw): 0.613000 | Accuracy (Rounded): 0.613
C = 0.001    -> Accuracy (Raw): 0.645000 | Accuracy (Rounded): 0.645

Best accuracy: (0.645). Smallest c score: 0.001
